## Load libraries

In [1]:
!pip install -q -U transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.7 MB/s eta 0:00:00


## Imports

In [3]:
import json
import glob
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

## Get model and tokenizer

In [4]:
model_id = "google/medgemma-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

## Build prompt

In [9]:
from IPython.display import display, Markdown

def run_extraction(target_drug, target_text):
    # Build the prompt string
    prompt = """SYSTEM: You are a medical informatics expert. Extract drug-drug interactions into a structured JSON list. The list should follow the following format: [{"@type": "","@precipitant": "","@precipitantCode": "","@effect": ""},...].\n\n"""
    prompt += f"### TASK\nDRUG: {target_drug}\nTEXT:\n{target_text}\n\nRESULT:"

    # Visual
    display(Markdown("---"))
    display(Markdown(f"**The following text is being sent to the model:**\n\n```text\n{prompt}\n```"))

    # Execute
    formatted_input = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n[" # Force JSON start
    inputs = tokenizer(formatted_input, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.01,
            do_sample=False
        )

    # Extract and format response
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    final_json = "[" + response

    display(Markdown(f"```json\n{final_json}\n```"))
    return final_json

## Sample it

In [10]:
# INPUT YOUR TEST CASE HERE
my_test_drug = "Warfarin" # @param {type: "string"}
my_test_text = "Warfarin metabolism is inhibited by fluconazole, leading to increased  prothrombin time. Concomitant administration of rifampin may decrease  warfarin effectiveness by inducing CYP enzymes. Patients should avoid  large amounts of leafy green vegetables (Vitamin K)." # @param {type: "string"}

# Execute
result = run_extraction(my_test_drug, my_test_text)

---

**The following text is being sent to the model:**

```text
SYSTEM: You are a medical informatics expert. Extract drug-drug interactions into a structured JSON list. The list should follow the following format: [{"@type": "","@precipitant": "","@precipitantCode": "","@effect": ""},...].

### TASK
DRUG: Warfarin
TEXT:
Warfarin metabolism is inhibited by fluconazole, leading to increased  prothrombin time. Concomitant administration of rifampin may decrease  warfarin effectiveness by inducing CYP enzymes. Patients should avoid  large amounts of leafy green vegetables (Vitamin K).

RESULT:
```

```json
[
  {
    "@type": "DrugDrugInteraction",
    "@precipitant": "fluconazole",
    "@precipitantCode": "ATC",
    "@effect": "increased prothrombin time"
  },
  {
    "@type": "DrugDrugInteraction",
    "@precipitant": "rifampin",
    "@precipitantCode": "ATC",
    "@effect": "decreased warfarin effectiveness"
  }
]

```

## API drug information testing

In [8]:
import requests
import json
from IPython.display import display, Markdown

def fetch_fda_interactions(drug_name):
    """
    Retrieves Section 7 (Drug Interactions) from the openFDA API.
    """
    # API endpoint
    base_url = "https://api.fda.gov/drug/label.json"

    # Build query
    params = {
        'search': f'openfda.brand_name:"{drug_name}" openfda.generic_name:"{drug_name}"',
        'limit': 1
    }

    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()

        if 'results' in data:
            label = data['results'][0]
            # 'drug_interactions' is the standard field for Section 7
            raw_text = label.get('drug_interactions', ["Section 7 not found in this label."])[0]

            return {
                "drug": drug_name.upper(),
                "text": raw_text,
                "source": "openFDA API"
            }
        return None
    except Exception as e:
        display(Markdown(f"**Error fetching {drug_name}:** {str(e)}"))
        return None

# Test
fetch_drug = "warfarin" # @param {type: "string"}
fetched = fetch_fda_interactions(fetch_drug)
if fetched:
    display(Markdown(f"### API Data: {fetched['drug']}"))
    display(Markdown(f"**Raw Section 7 Snippet:**\n> {fetched['text'][:500]}..."))

### API Data: WARFARIN

**Raw Section 7 Snippet:**
> 7 DRUG INTERACTIONS Concomitant use of drugs that increase bleeding risk, antibiotics, antifungals, botanical (herbal) products, and inhibitors and inducers of CYP2C9, 1A2, or 3A4. ( 7 ) Consult labeling of all concurrently used drugs for complete information about interactions with warfarin sodium or increased risks for bleeding. ( 7 ) 7.1 General Information Drugs may interact with warfarin sodium through pharmacodynamic or pharmacokinetic mechanisms. Pharmacodynamic mechanisms for drug intera...

In [ ]:
def test_new_drug_with_medgemma(drug_name):
    # Collect the FDA label
    collected_data = fetch_fda_interactions(drug_name)

    if not collected_data:
        display(Markdown(f"Could not find a label for **{drug_name}**"))
        return

    # Pass the collected text to our extraction engine
    display(Markdown(f"###Testing on: **{drug_name}**"))

    final_json = run_extraction(
        target_drug=collected_data['drug'],
        target_text=collected_data['text']
    )

    return final_json

# Try a drug that wasn't in the training set to see if it understands the pattern
test_drug = "warfarin" # @param {type: "string"}
test_case = test_new_drug_with_medgemma(test_drug)